# Basic

Evaluation of all candidate lists. Changes from the hop-based notebook:
- **distance score**: a token's depth is the number of primitive hop targets between the mapped concept and the token (uncovered contexts still count $D+1$);
- **semantic coverage**: still evaluated with $\lambda = 1$, so the transition matrix is the original one;
- greedy runs are read from `greedy_tree_candidates_prim_fd/`, only for the $\lambda$ values in `config.TokenizerParam.lambdas`.

Baseline candidate lists are unchanged (notebook 3 does not depend on $\lambda$ or on the status), but they are re-evaluated here because the distance score changed.

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, where src/ lives

import config_prim_fd as config             # found in decay_with_prim_fd/
import prim_fd_tokenizer as pf              # status-dependent coverage, selection and distance score

import polars as pl
import numpy as np
import tqdm
import os
import glob
import multiprocessing as mp

import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct
import src.graph_tokenizer_gd_tree_dev.utils as utils

In [3]:
Ks = config.TokenizerParam().Ks
rnd_iters = config.TokenizerParam().rnd_iters

In [4]:
id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()
is_primitive = pf.load_is_primitive(config.ProcessedGraph.concept_and_status, nodes=combined_subgraphs.nodes())

# only the lambdas of this experiment (the folder may still hold copies of older runs)
gd_tree_list = [f"{config.CandidateLists().path_greedy_tree}{lam}.parquet" for lam in config.TokenizerParam().lambdas]
missing = [f for f in gd_tree_list if not os.path.exists(f)]
assert not missing, f"run notebook 2 first, missing: {missing}"
baseline_list = glob.glob(config.CandidateLists().baseline_path + '/*.parquet')

D = config.TokenizerParam().max_dist_candidate
os.makedirs(config.Results().path, exist_ok=True)
D

status: 24,515 primitive / 51,449 fully defined (0 without status, treated as primitive)


9

In [5]:
all_candidates = {
    "greedy_tree_margin": utils.to_type_dict(gd_tree_list),
    "baseline": utils.to_type_dict(baseline_list),
}
for category, file_list in all_candidates.items():
    for file_type, file in file_list.items():
        print(f"Category: {category}, File Type: {file_type}, File: {file}")

Category: greedy_tree_margin, File Type: 0.0, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.0.parquet
Category: greedy_tree_margin, File Type: 0.1, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.1.parquet
Category: greedy_tree_margin, File Type: 0.2, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.2.parquet
Category: greedy_tree_margin, File Type: 0.3, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.3.parquet
Category: greedy_tree_margin, File Type: 0.4, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.4.parquet
Category: greedy_tree_margin, File Type: 0.5, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.5.parquet
Category: greedy_tree_margin, File Type: 0.6, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.6.parquet
Category: greedy_tree_margin, File Type: 0.7, File: D:/greedy_graph_data/greedy_tree_candidates_prim_fd/0.7.parquet
Category: greedy_tree_margin, File Type: 0.8, File: D:/greedy_graph_data

# shared evaluation structures

In [6]:
# Built once, shared by every task below (they depend on the graph only, not on T):
# A/node_to_idx: semantic-coverage transition at lambda = 1 -> w = 1 everywhere, identical to the original matrix
A, node_to_idx = pf.build_coverage_transition_status(combined_subgraphs, is_primitive, lam=1.0)
# adj: out-adjacency with each edge's target status -> context trees with primitive-hop depth (distance score)
adj = pf.build_out_adjacency_status(combined_subgraphs, is_primitive)

# baselines except k rnd

In [7]:
results = []
candidate_col = "token"
tasks = []
for category, file_list in all_candidates.items():
    if category == "greedy_tree_margin":
        continue
    for file_type, file in file_list.items():
        if file_type == "k_random_all_samples":
            continue

        df = pl.read_parquet(file)
        for k in Ks:
            candidates = df.head(k)[candidate_col].to_list()
            tasks.append(((category, file_type, k), candidates))

n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=pf._init_eval_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(pf._worker_eval, tasks, chunksize=4)
    for (category, file_type, k), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            **metrics,
        })

results_df = pl.DataFrame(results)
results_df.write_parquet(config.Results().perf_baseline_path)

100%|██████████| 273/273 [04:15<00:00,  1.07it/s]


# random K

In [8]:
results_k = []
candidate_col = "token"
category, file_type = "baseline", "k_random_all_samples"
df = pl.read_parquet(config.CandidateLists().k_random_all_samples)
tasks = [
    ((k, it), df.filter((pl.col("iter") == it) & (pl.col("k") == k))[candidate_col].to_list())
    for k in Ks
    for it in rnd_iters
]

n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=pf._init_eval_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(pf._worker_eval, tasks, chunksize=4)
    for (k, it), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results_k.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            "iter": it,
            **metrics,
        })

results_k_df = pl.DataFrame(results_k)
results_k_df.write_parquet(config.Results().perf_k_rdn_path)

100%|██████████| 780/780 [15:41<00:00,  1.21s/it] 


# greedy

In [9]:
results = []
candidate_col = "token"
tasks = []
for category, file_list in all_candidates.items():
    if category != "greedy_tree_margin":
        continue
    for file_type, file in file_list.items():
        df = pl.read_parquet(file)
        for k in Ks:
            candidates = df.head(k)[candidate_col].to_list()
            tasks.append(((category, file_type, k), candidates))
len(tasks)

780

In [10]:
n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=pf._init_eval_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(pf._worker_eval, tasks, chunksize=4)
    for (category, file_type, k), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            **metrics,
        })

results_df = pl.DataFrame(results)
results_df.write_parquet(config.Results().perf_greedy_tree_path)

100%|██████████| 780/780 [10:58<00:00,  1.18it/s]


# combine all performance

In [11]:
scores_metrics = ["conciseness", "distance_score", "uniqueness_entropy", "unk_rate", "uncovered_rate", "tree_complexity", "exact_rate", "unique_rate", "semantic_coverage"]
df_base = pl.read_parquet(config.Results().perf_baseline_path)
df_greedy_tree = pl.read_parquet(config.Results().perf_greedy_tree_path)
df_random = pl.read_parquet(config.Results().perf_k_rdn_path).group_by("category", "file_type","k").agg([pl.col(metric).mean() for metric in scores_metrics]).sort(by = "k")
df_perf_all = (pl.concat([df_base, df_greedy_tree, df_random], how="vertical")
               .rename({"unk_rate": "uncovered_branche_rate",
                        "uncovered_rate": "unknown_rate"}))
df_perf_all.write_parquet(config.Results().perf_all)
df_perf_all.write_csv(config.Results().perf_all_csv)

In [12]:
df_perf_all

category,file_type,k,conciseness,distance_score,uniqueness_entropy,uncovered_branche_rate,unknown_rate,tree_complexity,exact_rate,unique_rate,semantic_coverage
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""baseline""","""closeness_centrality""",12500,5.654821,0.919444,0.979413,0.0,0.0,4.404074,0.064789,0.840563,1.0
"""baseline""","""closeness_centrality""",13000,5.574041,0.920088,0.979894,0.0,0.0,4.368451,0.067627,0.84312,1.0
"""baseline""","""closeness_centrality""",13500,5.497508,0.920867,0.980543,0.0,0.0,4.333434,0.070596,0.846847,1.0
"""baseline""","""closeness_centrality""",14000,5.416576,0.922148,0.981567,0.0,0.0,4.301603,0.073889,0.855688,1.0
"""baseline""","""closeness_centrality""",10500,6.001387,0.914613,0.976048,0.000022,0.0,4.562969,0.052893,0.816901,1.0
…,…,…,…,…,…,…,…,…,…,…,…
"""baseline""","""k_random_all_samples""",17500,8.367849,0.653136,0.948827,0.679588,0.022349,6.458299,0.230169,0.70407,0.762918
"""baseline""","""k_random_all_samples""",18000,8.419973,0.66902,0.950734,0.656803,0.019205,6.432059,0.237281,0.710649,0.798867
"""baseline""","""k_random_all_samples""",18500,8.509014,0.685185,0.953218,0.635713,0.015479,6.314998,0.243766,0.718449,0.811554
